# Topic 4: Cloud Platforms — Databricks, EMR, Glue

> **Phase 7 → Week 12 → Topic 4**

---

## Platform Comparison

```
Platform        Managed?  Spark?   Extras                       Best for
──────────────  ────────  ───────  ───────────────────────────  ─────────────────────────
Databricks      Full      Yes      Delta Lake, Unity Catalog,   Teams wanting best-in-
                                   MLflow, notebooks, jobs UI    class managed Spark

AWS EMR         Partial   Yes      S3, Glue Catalog, IAM        AWS-native, cost control,
                                   EC2 spot instances            custom configs

AWS Glue        Full      Yes*     Serverless, Glue Catalog,    Serverless ETL, small
                (serverless)       DynamicFrame API              team, no cluster mgmt
                (*PySpark wrapper)

GCP Dataproc    Partial   Yes      GCS, BigQuery connector       GCP-native, hybrid

Azure Synapse   Partial   Yes      ADLS Gen2, Synapse Catalog    Azure-native, SQL + Spark

Local (Docker)  None      Yes      —                             Learning, CI/CD testing
```

---

## Interview Cheat Sheet

**Q: What is the difference between Databricks and AWS EMR?**
> Databricks is a fully managed Spark platform with auto-scaling clusters, collaborative notebooks, built-in Delta Lake, Unity Catalog for governance, and MLflow for ML tracking. You pay per DBU (Databricks Unit). EMR is AWS's managed Hadoop/Spark service — less opinionated, more configuration control, tighter AWS integration (Spot instances, IAM, Glue Catalog, S3). Databricks is better for teams wanting productivity; EMR is better for AWS-native architectures and cost optimization with Spot.

**Q: What is AWS Glue?**
> AWS Glue is a serverless ETL service. You write PySpark code (with some AWS extensions like DynamicFrame), and Glue provisions and manages the Spark cluster automatically — no cluster to configure. The Glue Data Catalog (schema registry) integrates with Athena, EMR, and Redshift. Best for: simple ETL without cluster management overhead. Not ideal for: complex tuning, streaming, or workloads needing precise resource control.

**Q: What is Databricks Delta Live Tables (DLT)?**
> Delta Live Tables is a declarative framework for building ETL pipelines in Databricks. You declare tables with `@dlt.table` decorators — DLT handles dependency resolution, incremental processing, data quality checks, and monitoring automatically. It abstracts away trigger logic, checkpointing, and orchestration. Best for: Medallion Architecture pipelines where each layer is a declared table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
print("This notebook is a reference guide — most code shows patterns/templates,")
print("not runnable in local mode (requires cloud credentials and infrastructure).")

## Part 1: AWS EMR

In [ ]:
print("""
AWS EMR — Key Patterns:
════════════════════════════════════════════════════════════════

1. Create a cluster and submit a job:

   aws emr create-cluster \\
     --name "ETL Cluster" \\
     --release-label emr-6.15.0 \\
     --applications Name=Spark \\
     --ec2-attributes KeyName=my-key \\
     --instance-groups \\
       InstanceGroupType=MASTER,InstanceCount=1,InstanceType=m5.xlarge \\
       InstanceGroupType=CORE,InstanceCount=4,InstanceType=m5.2xlarge \\
     --use-default-roles \\
     --log-uri s3://my-bucket/emr-logs/

   aws emr add-steps \\
     --cluster-id j-XXXX \\
     --steps Type=Spark,Name="My Job", \\
             ActionOnFailure=CONTINUE, \\
             Args=[\\
               --deploy-mode,cluster, \\
               --master,yarn, \\
               s3://my-bucket/scripts/etl_job.py, \\
               --env,prod, \\
               --date,2024-01-15\\
             ]

2. Read from S3 (automatic with EMR IAM role):
   df = spark.read.parquet("s3://my-bucket/data/orders/")
   df.write.parquet("s3://my-bucket/output/processed/")

3. Use Glue Data Catalog as metastore:
   --conf spark.sql.catalogImplementation=hive \\
   --conf hive.metastore.client.factory.class=\\
     com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory

4. Use Spot instances to cut cost 60-80%:
   InstanceGroupType=TASK,InstanceCount=8,InstanceType=m5.2xlarge,
   BidPrice=OnDemandPrice

5. S3-backed checkpoints (streaming):
   .option("checkpointLocation", "s3://my-bucket/checkpoints/my_query")

EMR vs Databricks:
  EMR:        More control, cheaper (Spot), tighter AWS integration
              Manual cluster sizing, no collaborative notebooks
  Databricks: Auto-scaling, collaborative, Delta + MLflow built-in
              Higher cost (DBU pricing on top of EC2)
════════════════════════════════════════════════════════════════
""")

## Part 2: AWS Glue

In [ ]:
print("""
AWS Glue — Key Patterns:
════════════════════════════════════════════════════════════════

Glue extends PySpark with DynamicFrame (schema-flexible DataFrame).

# Glue job boilerplate:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

args = getResolvedOptions(sys.argv, ['JOB_NAME', 'input_path', 'output_path'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# Read from Glue Catalog:
dyf = glueContext.create_dynamic_frame.from_catalog(
    database="my_db",
    table_name="orders"
)

# Convert to Spark DataFrame for full API:
df = dyf.toDF()

# ... your PySpark transformations ...

# Write back to S3 via Glue:
glueContext.write_dynamic_frame.from_options(
    frame=DynamicFrame.fromDF(result, glueContext, "result"),
    connection_type="s3",
    connection_options={"path": args['output_path']},
    format="parquet"
)

job.commit()  # mark job as successful

Glue-specific features:
  Bookmarks: track which S3 files already processed (like Spark checkpoints)
  DynamicFrame: handle semi-structured data, schema inconsistencies
  Glue Catalog: central schema registry for S3 data (used by Athena, EMR, Redshift)

Glue limitations:
  Cannot use spark.conf.set() for all configs (managed, not fully customizable)
  No streaming (use Kinesis Data Analytics instead)
  Cost: $0.44/DPU-hour (more expensive than EMR for large jobs)
════════════════════════════════════════════════════════════════
""")

## Part 3: Databricks

In [ ]:
print("""
Databricks — Key Patterns:
════════════════════════════════════════════════════════════════

Databricks provides: notebooks, jobs, clusters, Delta Lake, Unity Catalog,
MLflow, Workflows (Airflow-like), REST API.

1. Cluster types:
   All-purpose clusters:  interactive notebooks (expensive — keep shared)
   Job clusters:          ephemeral, spun up for a job, torn down after
                          (cheapest for production jobs)
   SQL Warehouses:        Photon engine, SQL-only, auto-stop, for BI tools

2. Delta Lake integration (built-in, no extra install):
   spark.read.table("my_catalog.my_schema.orders")   ← Unity Catalog
   df.write.format("delta").saveAsTable("silver.orders")
   spark.sql("OPTIMIZE silver.orders ZORDER BY (customer_id)")

3. Databricks Workflows (job orchestration):
   {"name": "Daily ETL",
    "tasks": [
      {"task_key": "bronze", "notebook_task": {"notebook_path": "/etl/bronze"}},
      {"task_key": "silver", "depends_on": [{"task_key": "bronze"}],
       "spark_python_task": {"python_file": "dbfs:/scripts/silver.py"}},
    ]}

4. Databricks CLI (submit jobs):
   databricks jobs run-now --job-id 12345
   databricks fs cp local_file.py dbfs:/scripts/

5. Unity Catalog (governance):
   Three-level namespace: catalog.schema.table
   my_catalog.silver.orders
   Fine-grained access control: row-level security, column masking

6. Delta Live Tables (DLT):
   import dlt
   from pyspark.sql import functions as F

   @dlt.table(comment="Raw orders from Kafka")
   def bronze_orders():
       return spark.readStream.format("kafka")...

   @dlt.table(comment="Validated orders")
   @dlt.expect("valid_amount", "amount > 0")
   def silver_orders():
       return dlt.read_stream("bronze_orders").filter(...)

Databricks cost optimization:
  Use spot instances for worker nodes (70% cheaper)
  Auto-terminate idle all-purpose clusters (--idle-time 30 min)
  Use job clusters instead of all-purpose for production
  Enable Photon (vectorized execution engine) for SQL workloads
════════════════════════════════════════════════════════════════
""")

## Part 4: Common S3/Cloud Patterns

In [ ]:
print("""
S3 + Spark Common Patterns:
════════════════════════════════════════════════════════════════

Reading from S3:
  df = spark.read.parquet("s3a://bucket/prefix/")   # s3a:// for Hadoop
  df = spark.read.json("s3://bucket/prefix/*.json") # glob patterns work

Writing to S3:
  df.write.mode("overwrite").parquet("s3a://bucket/output/")
  df.write.partitionBy("date").parquet("s3a://bucket/orders/")

S3 performance tips:
  1. Use s3a:// not s3:// or s3n:// (s3a is the fast, modern connector)
  2. Avoid small files: coalesce() before write
  3. S3 has no directory listing shortcuts — many small files = slow glob
  4. Use partitioning (partitionBy) for partition pruning
  5. Enable S3 committer for atomic writes:
     spark.conf.set("spark.hadoop.fs.s3a.committer.name", "magic")

Authentication:
  On EMR/Databricks: IAM role attached to instance — automatic
  Local testing:      export AWS_ACCESS_KEY_ID=...
                      export AWS_SECRET_ACCESS_KEY=...
  SparkSession config: 
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")

Glue Data Catalog as Hive Metastore:
  # All Spark SQL tables available via Glue Catalog
  spark.sql("SHOW DATABASES")
  spark.sql("SELECT * FROM my_database.orders LIMIT 10")
  # Athena can query the same tables (no data movement)

Cost optimization:
  Use Parquet + Snappy (columnar + compressed = fast reads, small files)
  Enable S3 Intelligent-Tiering for infrequent data
  Delete old staging data and temporary outputs regularly
  Use S3 Lifecycle rules to auto-delete temp prefixes after N days
════════════════════════════════════════════════════════════════
""")

## Exercises

1. You need to run a daily Spark job that processes 500 GB of S3 data. Compare the total cost estimate for: (a) EMR with On-Demand instances, (b) EMR with 80% Spot workers, (c) Databricks Job Cluster. What factors matter most for this decision?
2. What is the Glue Data Catalog and how does it differ from Spark's in-memory catalog? Why is it useful in a multi-service AWS architecture?
3. Explain the difference between `s3://`, `s3n://`, and `s3a://`. Which should you use and why?
4. Your Databricks job reads a Delta table using Unity Catalog. A colleague deletes a column from the table. Your job breaks. What schema evolution option would prevent this? What is the tradeoff?
5. When would you choose AWS Glue over EMR? Give three concrete criteria.